# Business Acumen & Metrics for Data Engineers
## KPIs, Business Metrics, Stakeholder Communication & Data-Driven Thinking

**What you'll learn:**
- Why business context makes you a 10x more effective DE
- Core business metrics by domain (SaaS, E-Commerce, FinTech, AdTech)
- Revenue metrics: ARR, MRR, ARPU, LTV, CAC, payback period
- Growth metrics: DAU/MAU, retention, churn, NRR, expansion revenue
- Product metrics: activation, engagement, conversion funnels
- Operational metrics: SLA, uptime, incident response, MTTR
- How to translate business questions into data models
- Stakeholder communication: dashboards, reports, data storytelling
- Building metrics layers (semantic layer, metrics store)
- Common business logic pitfalls in data pipelines
- Interview questions on business scenarios

**Estimated Time:** 6-8 hours

---

> The #1 thing that separates a senior DE from a junior is NOT technical skill --
> it's understanding WHY the data matters to the business and building systems
> that deliver the RIGHT metrics at the RIGHT time.

---

---
# Section 1: Why Business Acumen Matters for DEs

## The Difference It Makes

```
JUNIOR DE:
  "I built a pipeline that processes orders data into a table."

SENIOR DE:
  "I built a pipeline that computes daily MRR, churn rate, and NRR
   for the finance team's board reporting. It runs by 6 AM because
   the CFO reviews metrics at 7 AM. I partitioned by month for
   efficient period-over-period comparison, and added data quality
   checks because a $1M variance in ARR last quarter was caused by
   a duplicate subscription record."
```

## How Business Context Improves Your Work

| Without Business Context | With Business Context |
|-------------------------|----------------------|
| Build table, hand off | Build table optimized for actual query patterns |
| Generic data model | Model aligned with business process |
| SLA: "it runs daily" | SLA: "ready by 6 AM for CFO review" |
| Alert on pipeline failure | Alert on METRIC anomaly ($10K revenue drop) |
| Schema matches source | Schema matches business definitions |
| All data treated equally | Critical metrics have higher quality bars |

## The Business-Data Translation Skill

```
Business question: "Why did revenue drop last Tuesday?"
    ↓
Data translation: "Compare revenue by segment, channel, product
                   for Tuesday vs previous Tuesdays. Check for:
                   - Missing data (pipeline failure?)
                   - Specific segment drop (real trend?)
                   - Order count vs AOV change (volume vs price?)
                   - Payment failures (gateway issue?)"
    ↓
Pipeline requirement: Need a daily revenue breakdown table with
                      dimensions: date, segment, channel, product,
                      payment_method + metrics: revenue, orders,
                      AOV, payment_success_rate
```

---
# Section 2: Revenue & Financial Metrics

## SaaS Revenue Metrics

| Metric | Formula | What It Tells You |
|--------|---------|-------------------|
| **MRR** | Sum of all monthly subscription revenue | Current recurring revenue run-rate |
| **ARR** | MRR × 12 | Annual run-rate (for investors/board) |
| **New MRR** | MRR from new customers this month | Growth from acquisition |
| **Expansion MRR** | MRR increase from upgrades | Growth from existing customers |
| **Churned MRR** | MRR lost from cancellations | Revenue leakage |
| **Net New MRR** | New + Expansion - Churned | Net growth trajectory |
| **NRR (Net Revenue Retention)** | (Start MRR + Expansion - Churn) / Start MRR | >100% = growing without new customers |
| **ARPU** | Total Revenue / Active Users | Revenue per user efficiency |
| **LTV** | ARPU × Avg Customer Lifespan | Total value of a customer |
| **CAC** | Total Sales+Marketing Cost / New Customers | Cost to acquire one customer |
| **LTV:CAC** | LTV / CAC | >3x is healthy, <1x is unsustainable |
| **Payback Period** | CAC / Monthly Gross Profit per Customer | Months to recoup acquisition cost |

## E-Commerce Metrics

| Metric | Formula | What It Tells You |
|--------|---------|-------------------|
| **GMV** | Total value of all transactions | Platform scale |
| **Revenue** | GMV × Take Rate (platform fee) | Actual income |
| **AOV** | Revenue / Number of Orders | Average basket size |
| **Conversion Rate** | Purchases / Visitors | Funnel efficiency |
| **Cart Abandonment** | (Carts - Purchases) / Carts | Checkout friction |
| **Repeat Purchase Rate** | Customers with 2+ orders / Total Customers | Loyalty indicator |
| **Customer Acquisition Cost** | Marketing Spend / New Customers | Efficiency of growth |
| **Return Rate** | Returns / Orders | Product/fulfillment issues |

## FinTech Metrics

| Metric | Formula | What It Tells You |
|--------|---------|-------------------|
| **TPV** (Total Payment Volume) | Sum of all processed payments | Platform scale |
| **Take Rate** | Revenue / TPV | Margin per transaction |
| **Default Rate** | Defaulted Loans / Total Loans | Credit risk |
| **NPL Ratio** | Non-Performing Loans / Total Loans | Portfolio health |
| **Approval Rate** | Approved Applications / Total Applications | Underwriting efficiency |
| **Fraud Rate** | Fraudulent Transactions / Total | Security effectiveness |

In [ ]:
# Business metrics implementation in SQL/Python
print("IMPLEMENTING BUSINESS METRICS")
print("=" * 60)

print("""
-- MRR CALCULATION (SaaS):
-- Complex because of upgrades, downgrades, churn timing!

WITH subscription_mrr AS (
    SELECT
        date_trunc('month', billing_date) AS month,
        customer_id,
        plan_id,
        monthly_amount AS mrr
    FROM subscriptions
    WHERE status = 'active'
),
mrr_changes AS (
    SELECT
        curr.month,
        -- New MRR (first month for this customer)
        SUM(CASE WHEN prev.customer_id IS NULL THEN curr.mrr ELSE 0 END) AS new_mrr,
        -- Expansion (existing customer, higher MRR)
        SUM(CASE WHEN prev.mrr IS NOT NULL AND curr.mrr > prev.mrr
                 THEN curr.mrr - prev.mrr ELSE 0 END) AS expansion_mrr,
        -- Contraction (existing customer, lower MRR)
        SUM(CASE WHEN prev.mrr IS NOT NULL AND curr.mrr < prev.mrr
                 THEN prev.mrr - curr.mrr ELSE 0 END) AS contraction_mrr,
        -- Churned (was active last month, not this month)
        -- (handled via LEFT JOIN from previous month)
        SUM(curr.mrr) AS total_mrr
    FROM subscription_mrr curr
    LEFT JOIN subscription_mrr prev
        ON curr.customer_id = prev.customer_id
        AND curr.month = prev.month + INTERVAL '1 month'
    GROUP BY curr.month
)
SELECT
    month,
    total_mrr,
    new_mrr,
    expansion_mrr,
    contraction_mrr,
    total_mrr * 12 AS arr,
    -- NRR = (Start + Expansion - Contraction - Churn) / Start
    ROUND((total_mrr + expansion_mrr - contraction_mrr) * 100.0
          / LAG(total_mrr) OVER (ORDER BY month), 1) AS nrr_pct
FROM mrr_changes
ORDER BY month;

-- KEY GOTCHAS DEs MUST HANDLE:
-- 1. Mid-month upgrades: prorate or count full new amount?
-- 2. Paused subscriptions: churned or retained?
-- 3. Annual plans: divide by 12 for MRR (not count as single month)
-- 4. Refunds: subtract from month of refund or original month?
-- 5. Trials: include or exclude from MRR?
-- ALWAYS confirm business definitions with finance!
""")

# LTV calculation
print("LTV CALCULATION APPROACHES:")
print(f"  {'Method':<25} {'Formula':<40} {'Accuracy'}")
print(f"  {'-'*75}")
methods = [
    ("Simple", "ARPU x Avg Lifespan", "Low (assumes constant)"),
    ("Cohort-based", "Sum(cohort revenue) / cohort size", "Medium"),
    ("Predictive (ML)", "Model future revenue per customer", "High"),
    ("Historic", "Actual revenue from customer to date", "Backward-looking"),
]
for method, formula, accuracy in methods:
    print(f"  {method:<25} {formula:<40} {accuracy}")

---
# Section 3: Growth & Engagement Metrics

## User Growth Metrics

| Metric | Description | Healthy Range |
|--------|-------------|---------------|
| **DAU** | Daily Active Users | Depends on product |
| **WAU** | Weekly Active Users | Depends on product |
| **MAU** | Monthly Active Users | Depends on product |
| **DAU/MAU** | Stickiness ratio | >25% = good, >50% = excellent |
| **Growth Rate** | (New Users - Churned) / Total | >5% monthly = strong |
| **Viral Coefficient** | Invites × Conversion Rate | >1 = viral growth |
| **Activation Rate** | Users completing key action / Signups | >40% = good |

## Retention & Churn

```
RETENTION MATRIX (cohort analysis -- the most important chart):

         Month 0   Month 1   Month 2   Month 3   Month 6   Month 12
Jan '24:   100%      65%       52%       45%       30%       20%
Feb '24:   100%      68%       55%       48%       32%        -
Mar '24:   100%      70%       58%        -         -         -

Reading this:
  - Jan cohort: 100 users signed up, 65 still active after 1 month
  - Trend improving: Feb, Mar have better M1 retention (product improving)
  - "Flattening" around Month 3 = users who stay past M3 tend to stay forever
```

## Conversion Funnel

```
TYPICAL E-COMMERCE FUNNEL:

  Visit Website:     100,000  (100%)
       ↓
  View Product:       45,000  (45% -- browse rate)
       ↓
  Add to Cart:        12,000  (12% -- intent rate)
       ↓
  Start Checkout:      8,000  (8%)
       ↓
  Complete Purchase:   4,500  (4.5% -- conversion rate)

  KEY METRICS:
  - Overall CVR: 4.5% (purchases / visits)
  - Cart-to-Purchase: 37.5% (4500/12000)
  - Checkout Abandonment: 43.75% ((8000-4500)/8000)
  
  WHERE TO FOCUS: Biggest drop = biggest opportunity
  Here: Visit→Product (55% drop) suggests discovery problem
```

In [ ]:
# Funnel analysis SQL pattern
print("FUNNEL ANALYSIS PATTERN (SQL)")
print("=" * 60)

print("""
-- E-Commerce Funnel Query:
WITH funnel_steps AS (
    SELECT
        date_trunc('week', event_timestamp) AS week,
        COUNT(DISTINCT CASE WHEN event = 'page_view' THEN user_id END) AS visitors,
        COUNT(DISTINCT CASE WHEN event = 'product_view' THEN user_id END) AS product_views,
        COUNT(DISTINCT CASE WHEN event = 'add_to_cart' THEN user_id END) AS add_to_carts,
        COUNT(DISTINCT CASE WHEN event = 'checkout_start' THEN user_id END) AS checkouts,
        COUNT(DISTINCT CASE WHEN event = 'purchase' THEN user_id END) AS purchases
    FROM events
    GROUP BY date_trunc('week', event_timestamp)
)
SELECT
    week,
    visitors,
    purchases,
    ROUND(purchases * 100.0 / NULLIF(visitors, 0), 2) AS conversion_rate,
    ROUND(product_views * 100.0 / NULLIF(visitors, 0), 2) AS browse_rate,
    ROUND(purchases * 100.0 / NULLIF(add_to_carts, 0), 2) AS cart_to_purchase_rate,
    -- Week-over-week change
    ROUND((purchases - LAG(purchases) OVER (ORDER BY week)) * 100.0
          / NULLIF(LAG(purchases) OVER (ORDER BY week), 0), 1) AS wow_growth
FROM funnel_steps
ORDER BY week DESC;
""")

# Metrics layer concept
print("METRICS LAYER / SEMANTIC LAYER:")
print("""
  Problem: 5 teams calculate 'revenue' 5 different ways.
           Finance includes refunds, Sales doesn't.
           Marketing counts trials, Product doesn't.

  Solution: SINGLE SOURCE OF TRUTH for metric definitions.

  Tools:
    - dbt Metrics (dbt Semantic Layer)
    - Databricks Lakeview + AI/BI
    - LookML (Looker)
    - MetricFlow
    - Custom metrics table in Gold layer

  Pattern:
    gold.metrics_daily (
        date, metric_name, metric_value,
        dimension_1, dimension_2, ...
        -- e.g., date='2024-06-15', metric='revenue',
        --       segment='enterprise', channel='web', value=45000.00
    )
""")

---
# Section 4: Stakeholder Communication

## The DE Communication Framework

```
WHO are you talking to?         WHAT do they need?
┌──────────────────┐           ┌──────────────────┐
│ C-Suite (CEO/CFO)│           │ High-level KPIs  │
│                  │           │ Trends, not detail│
│ VP/Director      │           │ Segment insights │
│                  │           │ Action items     │
│ Data Analysts    │           │ Clean datasets   │
│                  │           │ Documentation    │
│ Other Engineers  │           │ API specs, schemas│
│                  │           │ Runbooks         │
└──────────────────┘           └──────────────────┘
```

## Data Storytelling Framework

1. **Context**: What's the situation? (MRR was $2M last month)
2. **Insight**: What changed? (MRR dropped 5% to $1.9M)
3. **Root cause**: Why? (Enterprise churn: 3 accounts worth $100K left)
4. **Impact**: So what? (If trend continues, miss Q4 target by $500K)
5. **Recommendation**: What should we do? (Focus retention on enterprise accounts at risk)

## Common Business Questions DEs Get

| Business Question | Data Requirement | Pipeline Implication |
|-------------------|-----------------|---------------------|
| "Why did revenue drop?" | Revenue by segment/day, refunds, cancellations | Dimensional model with daily grain |
| "Which customers will churn?" | Activity features, payment history, support tickets | Feature pipeline for ML |
| "What's our CAC by channel?" | Marketing spend + attribution + customer data | Multi-source join pipeline |
| "Are we hitting SLAs?" | Pipeline completion times, data freshness | Operational metrics pipeline |
| "What's the ROI of feature X?" | A/B test data, pre/post metrics | Experiment data pipeline |

In [ ]:
# Business metrics pitfalls
print("COMMON BUSINESS LOGIC PITFALLS IN DATA PIPELINES")
print("=" * 60)

pitfalls = [
    {
        "problem": "Double-counting revenue on refunds",
        "cause": "Refund creates a new negative transaction, but original not adjusted",
        "fix": "Net revenue = gross - refunds. Track separately. Confirm with finance which month the refund applies to.",
    },
    {
        "problem": "Counting trial users as 'active'",
        "cause": "No status filter, or unclear definition of 'customer'",
        "fix": "Define: customer = paid user with active subscription. Trials are separate metric.",
    },
    {
        "problem": "MRR calculation includes annual plans incorrectly",
        "cause": "Counting $12,000 annual plan as $12,000 MRR instead of $1,000",
        "fix": "MRR = annual_amount / 12 for annual plans. Confirm treatment of upfront payments.",
    },
    {
        "problem": "Timezone mismatch in daily metrics",
        "cause": "Events in UTC, business reports in PT/ET",
        "fix": "ALWAYS document timezone. Convert to business timezone for customer-facing metrics.",
    },
    {
        "problem": "Revenue attributed to wrong period",
        "cause": "Using payment_date vs order_date vs delivery_date",
        "fix": "Define: accrual-based (order date) vs cash-based (payment date). Finance decides.",
    },
    {
        "problem": "Churn rate denominator confusion",
        "cause": "Using end-of-period customers vs start-of-period",
        "fix": "Standard: Churned / Start-of-Period customers. Document formula in code comments.",
    },
    {
        "problem": "Vanity metrics masking real performance",
        "cause": "Reporting total signups instead of activated paying users",
        "fix": "Report actionable metrics: activation rate, paid conversion, revenue per user.",
    },
]

for i, p in enumerate(pitfalls, 1):
    print(f"\n  {i}. {p['problem']}")
    print(f"     Cause: {p['cause']}")
    print(f"     Fix: {p['fix']}")

---
# Section 5: Metrics by Industry

## AdTech / Marketing Analytics

| Metric | Formula | Meaning |
|--------|---------|---------|
| **CPM** | Cost / (Impressions / 1000) | Cost per thousand views |
| **CPC** | Cost / Clicks | Cost per click |
| **CPA** | Cost / Conversions | Cost per acquisition |
| **CTR** | Clicks / Impressions | Click-through rate |
| **ROAS** | Revenue from Ads / Ad Spend | Return on ad spend |
| **Attribution** | Which touchpoint gets credit | First-touch, last-touch, multi-touch |

## Marketplace / Platform

| Metric | Formula | Meaning |
|--------|---------|---------|
| **Liquidity** | Transactions / Active Listings | Supply-demand match |
| **Take Rate** | Platform Revenue / GMV | Platform margin |
| **Supply/Demand Ratio** | Sellers / Buyers | Market balance |
| **Time to Transaction** | Avg days from listing to sale | Market efficiency |
| **Repeat Rate (Both Sides)** | Returning buyers AND sellers | Platform stickiness |

## Healthcare / Insurance

| Metric | Formula | Meaning |
|--------|---------|---------|
| **Claims Ratio** | Claims Paid / Premiums Collected | Profitability |
| **PMPM** | Cost / (Members × Months) | Per-member per-month cost |
| **Denial Rate** | Denied Claims / Total Claims | Process efficiency |
| **Readmission Rate** | Readmissions within 30d / Discharges | Care quality |
| **HEDIS Scores** | Standardized quality measures | Plan quality rating |

In [ ]:
print("\nBUSINESS ACUMEN INTERVIEW QUESTIONS")
print("=" * 60)

questions = [
    {
        "q": "A PM says 'revenue dropped 10% yesterday.' How do you investigate?",
        "a": "1) Verify the data: check pipeline ran, no duplicates/missing data. 2) Segment: which products/regions/channels dropped? 3) Compare to patterns: is yesterday a holiday/weekend? 4) Check funnel: did traffic drop (top of funnel) or conversion (bottom)? 5) Check for one-time events: large refund, cancelled enterprise account? 6) Report findings with context: 'Revenue dropped $50K, primarily from enterprise segment in EMEA due to 2 churned accounts.'"
    },
    {
        "q": "How would you build a metrics layer for a growing startup?",
        "a": "1) Start with a 'gold.daily_metrics' table with date + dimensions + metric columns. 2) Define metrics in code (dbt metrics or custom SQL) with clear ownership. 3) Document every metric (formula, source, owner, timezone, caveats). 4) Version metric definitions (change to formula = new metric version, not silent change). 5) Build alerts on metric anomalies, not just pipeline failures. 6) Make metrics discoverable (data catalog, internal wiki)."
    },
    {
        "q": "What's the difference between MRR and revenue?",
        "a": "Revenue is total income recognized (GAAP/accounting). MRR is the RECURRING portion only, normalized to monthly. MRR excludes: one-time fees, professional services, setup charges. MRR includes: subscription fees, normalized annual plans (/12). Investors care about MRR because it's predictable; CFO cares about revenue because it's what gets reported."
    },
    {
        "q": "How do you handle conflicting metric definitions across teams?",
        "a": "1) Acknowledge both are valid for their context. 2) Create a SINGLE canonical definition in the metrics layer (blessed by finance/leadership). 3) Name variants explicitly: 'revenue_gross', 'revenue_net', 'revenue_recognized'. 4) Document in data catalog with owner and use case for each. 5) Deprecate unofficial sources gradually (don't break workflows)."
    },
    {
        "q": "How would you design a data model for churn prediction?",
        "a": "Features: recency (days since last activity), frequency (sessions/month trend), monetary (MRR, LTV to date), engagement (feature usage), support (ticket count, sentiment), contract (days until renewal, plan tier). Label: did_churn_within_30d (binary). Grain: one row per customer per snapshot date. Update daily. Serve from feature store for real-time scoring."
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─'*60}")
    print(f"  {qa['a']}")

---
# Summary

## Business Acumen Cheat Sheet for DEs

| Skill | What It Means | How to Develop |
|-------|--------------|----------------|
| Metric literacy | Know what KPIs mean and how they're calculated | Ask finance/product to explain their dashboards |
| Business translation | Convert questions to data requirements | Practice: "What data do I need to answer this?" |
| Stakeholder empathy | Understand what keeps them up at night | Attend business reviews, read board decks |
| Impact quantification | Express your work in business terms | "Reduced report delay → earlier decisions → $X saved" |
| Priority judgment | Know which metrics are mission-critical | If this metric is wrong, who gets fired? |

## The Golden Rules

1. **Ask WHY before HOW** — Understand the business question before building the pipeline
2. **Revenue is king** — If your pipeline affects revenue calculation, it's critical path
3. **Define before you build** — Get metric definitions signed off in writing
4. **Freshness = trust** — Stale data erodes confidence in the entire platform
5. **Context over precision** — A directionally correct answer today beats a perfect answer next week

---
*Business Acumen & Metrics for Data Engineers*